# Rewired-Delaunay Robustness Sweep

Evaluates the frozen best model (`model-checkpoints/model-best.ckpt`) on Delaunay graphs whose edges have been rewired to random long-range links with probability `p`, at N = 800 and N = 1600. No retraining.

Run the sweep first, from the repo root:

```bash
python -m src.experiments.train_bfs --cfg src/configs/bfs/rewired-delaunay-sweep.yml
```

That writes `results/bfs/rewired-delaunay-sweep/<seed>.csv`, which this notebook reads.

**Sanity checks** (measured in the same run, as the `delaunay_*` / `er_*` / `ws_*` reference bins):

- `p = 0` should match the `delaunay` bins — the generator is a literal no-op at `p = 0`, so any gap is sampling noise only.
- `p → 1` destroys all local structure. Note this approaches an **ER-like random graph**, not a Watts-Strogatz graph — WS keeps high clustering because it uses *low* `p`. The `er` reference is the meaningful high-`p` target; `ws` is plotted for context but sits off this axis.

In [ ]:
import glob
import os
import re

import matplotlib.pyplot as plt
import pandas as pd

RESULTS_DIR = "../results/bfs/rewired-delaunay-sweep"
METRICS = ["graph_accuracy", "node_accuracy"]

csv_paths = sorted(glob.glob(os.path.join(RESULTS_DIR, "*.csv")))
if not csv_paths:
    raise FileNotFoundError(
        f"No results in {RESULTS_DIR}. Run the sweep config first (see the cell above)."
    )

# One row per seed; the sweep is single-seed but averaging is harmless.
raw = pd.concat([pd.read_csv(p) for p in csv_paths], ignore_index=True)
print(f"Loaded {len(csv_paths)} result file(s): {[os.path.basename(p) for p in csv_paths]}")
raw.filter(like="graph_accuracy").T.head(12)

In [ ]:
# Columns look like `test/graph_accuracy/rd_800_p0050`, where p is in permille.
SWEEP_RE = re.compile(r"^test/(?P<metric>\w+)/rd_(?P<n>\d+)_p(?P<permille>\d+)$")
REF_RE = re.compile(r"^test/(?P<metric>\w+)/(?P<family>delaunay|er|ws)_(?P<n>\d+)$")

sweep_rows, ref_rows = [], []
for col in raw.columns:
    value = raw[col].mean()
    if (m := SWEEP_RE.match(col)) and m["metric"] in METRICS:
        sweep_rows.append({"metric": m["metric"], "n": int(m["n"]),
                           "p": int(m["permille"]) / 1000, "value": value})
    elif (m := REF_RE.match(col)) and m["metric"] in METRICS:
        ref_rows.append({"metric": m["metric"], "n": int(m["n"]),
                         "family": m["family"], "value": value})

sweep = pd.DataFrame(sweep_rows).sort_values(["metric", "n", "p"])
refs = pd.DataFrame(ref_rows)

sweep.pivot_table(index="p", columns=["metric", "n"], values="value")

In [ ]:
N_STYLES = {800: ("o", "#2b6cb0"), 1600: ("s", "#c05621")}
REF_STYLES = {"delaunay": ":", "er": "--", "ws": "-."}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric in zip(axes, METRICS):
    for n, (marker, color) in N_STYLES.items():
        series = sweep[(sweep.metric == metric) & (sweep.n == n)]
        if series.empty:
            continue
        ax.plot(series.p, series.value, marker=marker, color=color,
                label=f"rewired Delaunay, N={n}", zorder=3)

        for family, linestyle in REF_STYLES.items():
            ref = refs[(refs.metric == metric) & (refs.n == n) & (refs.family == family)]
            if not ref.empty:
                ax.axhline(ref.value.iloc[0], linestyle=linestyle, color=color,
                           alpha=0.45, linewidth=1.2, zorder=1,
                           label=f"{family}_{n} reference")

    # symlog keeps p=0 on the axis while spreading the dense low-p samples.
    ax.set_xscale("symlog", linthresh=0.01)
    ax.set_xlabel("rewiring fraction $p$")
    ax.set_ylabel(metric.replace("_", " "))
    ax.set_title(f"{metric.replace('_', ' ').title()} vs rewiring fraction")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc="best")

axes[0].set_ylim(-0.05, 1.05)  # graph accuracy is all-or-nothing; show the full range

fig.suptitle("Frozen best model on rewired Delaunay graphs (no retraining)", fontsize=13)
fig.tight_layout()
plt.show()

## Sanity check

`p = 0` against the `delaunay` reference bins. The generator short-circuits at `p = 0`, so these differ only by which graphs were sampled — a gap here means sampling noise (or a bug), not a structural effect.

In [ ]:
check = []
for metric in METRICS:
    for n in sorted(sweep.n.unique()):
        at_zero = sweep[(sweep.metric == metric) & (sweep.n == n) & (sweep.p == 0.0)]
        ref = refs[(refs.metric == metric) & (refs.n == n) & (refs.family == "delaunay")]
        if at_zero.empty or ref.empty:
            continue
        check.append({
            "metric": metric, "N": n,
            "p=0": at_zero.value.iloc[0],
            "delaunay ref": ref.value.iloc[0],
            "abs diff": abs(at_zero.value.iloc[0] - ref.value.iloc[0]),
        })

pd.DataFrame(check).round(4)

In [ ]:
# High-p end against the reference families, to see which one it actually lands on.
tail = []
for metric in METRICS:
    for n in sorted(sweep.n.unique()):
        at_one = sweep[(sweep.metric == metric) & (sweep.n == n) & (sweep.p == 1.0)]
        if at_one.empty:
            continue
        row = {"metric": metric, "N": n, "p=1": at_one.value.iloc[0]}
        for family in REF_STYLES:
            ref = refs[(refs.metric == metric) & (refs.n == n) & (refs.family == family)]
            row[f"{family} ref"] = ref.value.iloc[0] if not ref.empty else float("nan")
        tail.append(row)

pd.DataFrame(tail).round(4)